# CTGAN training, CaNN reconstruction and privacy gates


In [ ]:
import json
import os
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    PACKAGE_ROOT = PACKAGE_ROOT.parent
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    raise FileNotFoundError("geoctgan")
os.chdir(PACKAGE_ROOT)
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from geoctgan.config import Config

def input_path(label, required=True):
    raw = input(label + ": ").strip().strip('"').strip("'")
    if not raw:
        if not required:
            return None
        raise ValueError(label)
    path = Path(raw).expanduser().resolve()
    if required and not path.exists():
        raise FileNotFoundError(label)
    return path

def stored_path(path):
    return str(Path(path).resolve())

@contextmanager
def configuration_file(values):
    with TemporaryDirectory(prefix="geoctgan_") as directory:
        path = Path(directory) / "configuration.json"
        path.write_text(json.dumps(values, indent=2) + "\n", encoding="utf-8")
        yield path


## Study data and selected configuration


In [ ]:
EVENT_ZIP = input_path("Toronto event archive")
ROAD_ZIP = input_path("Toronto road-network archive")
SELECTED_CONFIG = input_path("Selected configuration JSON")
OUTPUT_PARENT = input_path("Output directory", required=False) or (PACKAGE_ROOT / "Output")
RUN_ROOT = OUTPUT_PARENT / "Toronto"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
EVENT_MEMBER = "MCI_OPEN_DATA.shp"
NETWORK_MEMBER = "RoadNetworkExport/toronto_network.gpkg"
NETWORK_LAYER = "roads_edges"
MODEL_SEEDS = [42, 52, 62]


## Multi-seed generation


In [ ]:
from geoctgan.pipeline import run_pipeline

definitive_values = {
    "BASE_DIR": stored_path(EVENT_ZIP.parent),
    "OUTPUT_DIR": stored_path(RUN_ROOT),
    "ZIP_EVENTS": stored_path(EVENT_ZIP),
    "ZIP_NETWORK": stored_path(ROAD_ZIP),
    "EVENTS_MEMBER": EVENT_MEMBER,
    "NETWORK_MEMBER": NETWORK_MEMBER,
    "NETWORK_LAYER": NETWORK_LAYER,
    "SELECTED_CONFIG_PATH": stored_path(SELECTED_CONFIG),
    "REQUIRE_SELECTED_CONFIG": True,
    "EPOCHS": 300,
    "ENABLE_GPU": True,
    "DEFINITIVE_SEEDS": MODEL_SEEDS,
    "BENCHMARK_SEEDS": MODEL_SEEDS,
    "PUBLIC_RELEASE_ENABLED": True,
    "PUBLIC_RELEASE_CAPACITY_MODE": "upper_quota",
    "ALLOW_EXACT_QUOTA_PUBLIC_RELEASE": False,
    "RESUME_TRAINING": True,
    "RESUME_GENERATION": True,
    "RESUME_CANN": True,
}
with configuration_file(definitive_values) as DEFINITIVE_CONFIG:
    generation = run_pipeline(str(DEFINITIVE_CONFIG), seeds=MODEL_SEEDS, run_dir=RUN_ROOT / "Generation")
GENERATION_SUMMARY = RUN_ROOT / "Generation" / "Metadata" / "run_summary.json"
